In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np

def create_dataset_csv_from_directory(img_dir, output_csv='all_531_datasets.csv'):
    """
    Scan image directory and create CSV with all image files
    """

    print("="*70)
    print("CREATING CSV FILE FROM IMAGE DIRECTORY")
    print("="*70)

    image_data = []
    supported_formats = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']

    print(f"\nScanning directory: {img_dir}")

    # Walk through directory
    for root, dirs, files in os.walk(img_dir):
        for filename in files:
            # Check if it's an image file
            if any(filename.lower().endswith(fmt) for fmt in supported_formats):

                filepath = os.path.join(root, filename)

                try:
                    # Open image to get dimensions
                    img = Image.open(filepath)
                    width, height = img.size

                    # Extract dataset info from path or filename
                    # Adjust based on your directory structure
                    relative_path = os.path.relpath(filepath, img_dir)
                    dataset_name = relative_path.split(os.sep)[0] if os.sep in relative_path else 'main'

                    image_data.append({
                        'filename': os.path.relpath(filepath, img_dir),
                        'full_path': filepath,
                        'dataset_name': dataset_name,
                        'width': width,
                        'height': height,
                        'format': img.format
                    })

                    if len(image_data) % 1000 == 0:
                        print(f"  Processed {len(image_data)} images...")

                except Exception as e:
                    print(f"  Error processing {filename}: {e}")

    # Create DataFrame
    df = pd.DataFrame(image_data)

    print(f"\n✓ Found {len(df)} images")
    print(f"\nDataset distribution:")
    print(df['dataset_name'].value_counts())

    # Save to CSV
    df.to_csv(output_csv, index=False)
    print(f"\n✓ CSV saved to: {output_csv}")

    return df

# Usage
IMG_DIR = 'handwriting-project-2026\Combined'
all_data_df = create_dataset_csv_from_directory(IMG_DIR, 'all_531_datasets.csv')


CREATING CSV FILE FROM IMAGE DIRECTORY

Scanning directory: handwriting-project-2026\Combined

✓ Found 531 images

Dataset distribution:
dataset_name
main    531
Name: count, dtype: int64

✓ CSV saved to: all_531_datasets.csv


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split


print("="*70)
print("COMPLETE WORKFLOW: SPLIT → TRAIN → PREDICT QUALITY")
print("="*70)


# ============= STEP 1: Load CSV and Calculate Blur Scores =============
print("\n[STEP 1] Loading CSV and calculating blur scores...")

all_data_df = pd.read_csv('all_531_datasets.csv')
print(f"Total images loaded: {len(all_data_df)}")

def calculate_blur_score(img_path, base_dir='handwriting-project-2026/Combined'):
    """Calculate Laplacian variance blur score"""
    try:
        # Handle both relative and full paths
        if not os.path.isabs(img_path):
            full_path = os.path.join(base_dir, img_path)
        else:
            full_path = img_path

        img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        return cv2.Laplacian(img, cv2.CV_64F).var()
    except Exception as e:
        print(f"Error calculating blur for {img_path}: {e}")
        return None

# Calculate blur scores for all images
print("Calculating blur scores (this may take a few minutes)...")
all_data_df['blur_score'] = all_data_df['filename'].apply(
    lambda x: calculate_blur_score(x)
)

# Remove images with failed blur calculation
all_data_df = all_data_df[all_data_df['blur_score'].notna()].reset_index(drop=True)
print(f"Images with valid blur scores: {len(all_data_df)}")
print(f"\nBlur score statistics:")
print(all_data_df['blur_score'].describe())


# ============= STEP 2: Assign Quality Labels =============
print("\n[STEP 2] Assigning quality labels...")

def assign_quality_label(blur_score):
    """Assign quality category based on blur score thresholds"""
    if blur_score < 1000:
        return 0  # Low quality
    elif blur_score < 3000:
        return 1  # Medium quality
    elif blur_score < 5000:
        return 2  # Good quality
    else:
        return 3  # Excellent quality

all_data_df['quality_label'] = all_data_df['blur_score'].apply(assign_quality_label)

print("\nQuality distribution in full dataset:")
print(all_data_df['quality_label'].value_counts().sort_index())
quality_names = {0: 'Low', 1: 'Medium', 2: 'Good', 3: 'Excellent'}
for label, name in quality_names.items():
    count = (all_data_df['quality_label'] == label).sum()
    print(f"  {name}: {count} ({100*count/len(all_data_df):.2f}%)")


# ============= STEP 3: Split into Train/Val/Test =============
print("\n[STEP 3] Splitting into train/val/test sets...")

# First split: 80% train+val, 20% test
train_val_df, test_df = train_test_split(
    all_data_df,
    test_size=0.2,
    stratify=all_data_df['quality_label'],
    random_state=42
)

# Second split: 80% train, 20% val (of the train_val set)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    stratify=train_val_df['quality_label'],
    random_state=42
)

print(f"\nSplit sizes:")
print(f"  Train: {len(train_df)} samples ({100*len(train_df)/len(all_data_df):.1f}%)")
print(f"  Val:   {len(val_df)} samples ({100*len(val_df)/len(all_data_df):.1f}%)")
print(f"  Test:  {len(test_df)} samples ({100*len(test_df)/len(all_data_df):.1f}%)")

# Save split CSVs
train_df.to_csv('train_set.csv', index=False)
val_df.to_csv('val_set.csv', index=False)
test_df.to_csv('test_set.csv', index=False)
print("\n✓ Saved: train_set.csv, val_set.csv, test_set.csv")


# ============= STEP 4: Dataset Class =============
class QualityDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        try:
            filename = self.data.iloc[idx]['filename']
            img_path = os.path.join(self.img_dir, filename)
            image = Image.open(img_path).convert('RGB')
            label = int(self.data.iloc[idx]['quality_label'])

            if self.transform:
                image = self.transform(image)

            return image, label
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            return torch.zeros(3, 224, 224), 0


# ============= STEP 5: Data Transforms =============
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# ============= STEP 6: Create DataLoaders =============
IMG_DIR = 'handwriting-project-2026/Combined'

train_dataset = QualityDataset(train_df, IMG_DIR, transform=train_transform)
val_dataset = QualityDataset(val_df, IMG_DIR, transform=test_transform)
test_dataset = QualityDataset(test_df, IMG_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"\n{'='*70}")
print("Datasets created:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


# ============= STEP 7: Model Architecture =============
class QualityCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(QualityCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Global pooling
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ============= STEP 8: Training Function =============
def train_model(model, train_loader, val_loader, epochs=100, device='cuda'):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f"\n{'='*70}")
    print("TRAINING CNN MODEL")
    print(f"{'='*70}")

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0


        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

        train_acc = 100. * train_correct / train_total
        train_loss = train_loss / len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / len(val_loader)

        scheduler.step(val_loss)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_quality_model.pth')
            print(f'Epoch {epoch+1}/{epochs} - ✓ NEW BEST! Val Acc: {val_acc:.2f}%')
        else:
            print(f'Epoch {epoch+1}/{epochs} - Train: {train_acc:.2f}%, Val: {val_acc:.2f}%')

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

    print(f"\n✓ Training Complete! Best Val Acc: {best_val_acc:.2f}%")
    return history, best_val_acc


# ============= STEP 9: Train the Model =============
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

model = QualityCNN(num_classes=4)
history, best_val_acc = train_model(model, train_loader, val_loader, epochs=100, device=device)


# ============= STEP 10: Apply Model to ALL 531 Datasets =============
print(f"\n{'='*70}")
print("APPLYING MODEL TO ALL 531 DATASETS")
print(f"{'='*70}")

def predict_quality_for_all(model, all_data_df, img_dir, device='cuda'):
    """Apply trained quality classifier to entire dataset"""

    state_dict = torch.load('best_quality_model.pth', weights_only=True)
    model.load_state_dict(state_dict)

    model = model.to(device)
    model.eval()

    # Create inference dataset
    class InferenceDataset(Dataset):
        def __init__(self, dataframe, img_dir, transform):
            self.data = dataframe.reset_index(drop=True)
            self.img_dir = img_dir
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            try:
                filename = self.data.iloc[idx]['filename']
                img_path = os.path.join(self.img_dir, filename)
                image = Image.open(img_path).convert('RGB')

                if self.transform:
                    image = self.transform(image)

                return image, idx
            except Exception as e:
                return torch.zeros(3, 224, 224), idx

    inference_dataset = InferenceDataset(all_data_df, img_dir, transform=test_transform)
    inference_loader = DataLoader(inference_dataset, batch_size=32, shuffle=False, num_workers=0)

    all_predictions = np.zeros(len(all_data_df), dtype=int)
    all_probabilities = np.zeros((len(all_data_df), 4), dtype=float)

    print(f"Predicting quality for {len(all_data_df)} images...")

    with torch.no_grad():
        for batch_idx, (images, indices) in enumerate(inference_loader):
            images = images.to(device)
            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            all_predictions[indices.numpy()] = predicted.cpu().numpy()
            all_probabilities[indices.numpy()] = probs.cpu().numpy()

            if (batch_idx + 1) % 50 == 0:
                print(f"  Processed {(batch_idx + 1) * 32} / {len(all_data_df)} images")

    return all_predictions, all_probabilities

# Run inference on ALL images
predictions, probabilities = predict_quality_for_all(model, all_data_df, IMG_DIR, device)

# Add predictions to dataframe
all_data_df['predicted_quality'] = predictions
all_data_df['quality_confidence'] = probabilities.max(axis=1)

quality_map = {0: 'Low', 1: 'Medium', 2: 'Good', 3: 'Excellent'}
all_data_df['quality_category'] = all_data_df['predicted_quality'].map(quality_map)

print(f"\n✓ Quality prediction complete!")


# ============= STEP 11: Quality Analysis and Filtering =============
print(f"\n{'='*70}")
print("QUALITY ANALYSIS FOR PAPER")
print(f"{'='*70}")

print("\nPredicted Quality Distribution (All 531 Datasets):")
quality_dist = all_data_df['quality_category'].value_counts()
for category in ['Low', 'Medium', 'Good', 'Excellent']:
    if category in quality_dist:
        count = quality_dist[category]
        percentage = 100 * count / len(all_data_df)
        print(f"  {category:12s}: {count:6d} images ({percentage:5.2f}%)")

# Apply quality filtering
QUALITY_THRESHOLD = 2  # Keep 'Good' and 'Excellent'
CONFIDENCE_THRESHOLD = 0.7

filtered_df = all_data_df[
    (all_data_df['predicted_quality'] >= QUALITY_THRESHOLD) &
    (all_data_df['quality_confidence'] >= CONFIDENCE_THRESHOLD)
]

print(f"\nQuality Filtering Results:")
print(f"  Original images: {len(all_data_df)}")
print(f"  After filtering (≥Good, conf≥0.7): {len(filtered_df)}")
print(f"  Retention rate: {100 * len(filtered_df) / len(all_data_df):.2f}%")


# ============= STEP 12: Save All Results =============
print(f"\n{'='*70}")
print("SAVING RESULTS")
print(f"{'='*70}")

# Save complete dataset with predictions
all_data_df.to_csv('all_531_datasets_with_quality.csv', index=False)
print("✓ all_531_datasets_with_quality.csv")

# Save filtered high-quality dataset
filtered_df.to_csv('filtered_high_quality_dataset.csv', index=False)
print("✓ filtered_high_quality_dataset.csv")

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv('training_history.csv', index=False)
print("✓ training_history.csv")

# Save comprehensive report
import json
report = {
    'total_images': len(all_data_df),
    'quality_distribution': all_data_df['quality_category'].value_counts().to_dict(),
    'filtered_images': len(filtered_df),
    'retention_rate': f"{100 * len(filtered_df) / len(all_data_df):.2f}%",
    'best_val_accuracy': f"{best_val_acc:.2f}%",
    'quality_threshold': quality_map[QUALITY_THRESHOLD],
    'confidence_threshold': CONFIDENCE_THRESHOLD
}

with open('quality_analysis_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print("✓ quality_analysis_report.json")


# ============= STEP 13: Paper Summary =============
print(f"\n{'='*70}")
print("SUMMARY FOR PAPER - QUALITY REQUIREMENT SECTION")
print(f"{'='*70}")

print(f"""
We developed an automated quality assessment system using a CNN classifier
trained on {len(train_df)} images with known blur scores. The model achieved
{best_val_acc:.2f}% validation accuracy in categorizing images into four
quality levels: Low, Medium, Good, and Excellent.

Applying this classifier to all {len(all_data_df)} images across 531 datasets:
- Low Quality: {(all_data_df['predicted_quality']==0).sum()} images
- Medium Quality: {(all_data_df['predicted_quality']==1).sum()} images
- Good Quality: {(all_data_df['predicted_quality']==2).sum()} images
- Excellent Quality: {(all_data_df['predicted_quality']==3).sum()} images

For the final dataset, we retained only images classified as 'Good' or
'Excellent' with confidence ≥70%, resulting in {len(filtered_df)} high-quality
images ({100*len(filtered_df)/len(all_data_df):.1f}% retention rate).
""")

print(f"\n{'='*70}")
print("WORKFLOW COMPLETE!")
print(f"{'='*70}")


COMPLETE WORKFLOW: SPLIT → TRAIN → PREDICT QUALITY

[STEP 1] Loading CSV and calculating blur scores...
Total images loaded: 531
Calculating blur scores (this may take a few minutes)...
Images with valid blur scores: 531

Blur score statistics:
count      531.000000
mean      3268.799851
std       2438.882010
min          9.724937
25%        795.160729
50%       4110.863217
75%       5061.861892
max      13865.076692
Name: blur_score, dtype: float64

[STEP 2] Assigning quality labels...

Quality distribution in full dataset:
quality_label
0    145
1     87
2    159
3    140
Name: count, dtype: int64
  Low: 145 (27.31%)
  Medium: 87 (16.38%)
  Good: 159 (29.94%)
  Excellent: 140 (26.37%)

[STEP 3] Splitting into train/val/test sets...

Split sizes:
  Train: 339 samples (63.8%)
  Val:   85 samples (16.0%)
  Test:  107 samples (20.2%)

✓ Saved: train_set.csv, val_set.csv, test_set.csv

Datasets created:
  Train: 339 samples
  Val: 85 samples
  Test: 107 samples

Using device: cuda

TRAINI

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, test_loader, device='cuda'):
    model.load_state_dict(torch.load('best_quality_model.pth'))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = 100.0 * correct / total
    print(f"\nTest Accuracy: {test_acc:.2f}%")
    print(f"Correct: {correct}/{total}")

    print("\nClassification report:")
    print(classification_report(all_labels, all_preds,
          target_names=['Low','Medium','Good','Excellent']))

    print("\nConfusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return test_acc

# After training:
test_acc = evaluate_model(model, test_loader, device=device)


C:\Users\IISER\AppData\Local\Temp\ipykernel_12968\4171286318.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_quality_model.pth'))



Test Accuracy: 85.98%
Correct: 92/107

Classification report:
              precision    recall  f1-score   support

         Low       0.96      0.90      0.93        29
      Medium       0.75      0.83      0.79        18
        Good       0.93      0.81      0.87        32
   Excellent       0.78      0.89      0.83        28

    accuracy                           0.86       107
   macro avg       0.86      0.86      0.85       107
weighted avg       0.87      0.86      0.86       107


Confusion matrix:
[[26  3  0  0]
 [ 1 15  0  2]
 [ 0  1 26  5]
 [ 0  1  2 25]]


In [ ]:
import os
import json
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


print("="*70)
print("BINARY QUALITY CLASSIFICATION - COMPLETE WORKFLOW")
print("="*70)


# ====================== CONFIG ======================
IMG_DIR = 'handwriting-project-2026/Combined'
CSV_PATH = 'all_531_datasets.csv'
BATCH_SIZE = 16
EPOCHS = 100
LR = 0.001
BLUR_THRESHOLD = 3000  # Low: <3000, High: >=3000
RANDOM_STATE = 42


# ====================== STEP 1: LOAD CSV ======================
print("\n[STEP 1] Loading CSV...")
all_data_df = pd.read_csv(CSV_PATH)
print(f"Total images: {len(all_data_df)}")


# ====================== STEP 2: BLUR SCORES ======================
print("\n[STEP 2] Calculating blur scores...")

def calculate_blur_score(img_path, base_dir=IMG_DIR):
    try:
        if not os.path.isabs(img_path):
            full_path = os.path.join(base_dir, img_path)
        else:
            full_path = img_path
        img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        return cv2.Laplacian(img, cv2.CV_64F).var()
    except Exception as e:
        return None

if 'blur_score' not in all_data_df.columns:
    print("Computing blur scores for all images...")
    all_data_df['blur_score'] = all_data_df['filename'].apply(calculate_blur_score)
else:
    print("Using existing blur scores.")

# Remove invalid
all_data_df = all_data_df[all_data_df['blur_score'].notna()].reset_index(drop=True)
print(f"Valid images: {len(all_data_df)}")

print("\nBlur score statistics:")
print(all_data_df['blur_score'].describe())


# ====================== STEP 3: BINARY LABELS ======================
print(f"\n[STEP 3] Creating binary labels (threshold={BLUR_THRESHOLD})...")

# Binary: 0 = Low Quality (<3000), 1 = High Quality (>=3000)
all_data_df['binary_label'] = (all_data_df['blur_score'] >= BLUR_THRESHOLD).astype(int)

print("\nBinary distribution (full dataset):")
low_count = (all_data_df['binary_label'] == 0).sum()
high_count = (all_data_df['binary_label'] == 1).sum()
print(f"  Low Quality (<{BLUR_THRESHOLD}):  {low_count:4d} ({100*low_count/len(all_data_df):.2f}%)")
print(f"  High Quality (>={BLUR_THRESHOLD}): {high_count:4d} ({100*high_count/len(all_data_df):.2f}%)")


# ====================== STEP 4: TRAIN/VAL/TEST SPLIT ======================
print("\n[STEP 4] Splitting into train/val/test...")

train_val_df, test_df = train_test_split(
    all_data_df,
    test_size=0.2,
    stratify=all_data_df['binary_label'],
    random_state=RANDOM_STATE
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    stratify=train_val_df['binary_label'],
    random_state=RANDOM_STATE
)

print(f"Train: {len(train_df)} ({100*len(train_df)/len(all_data_df):.1f}%)")
print(f"Val:   {len(val_df)} ({100*len(val_df)/len(all_data_df):.1f}%)")
print(f"Test:  {len(test_df)} ({100*len(test_df)/len(all_data_df):.1f}%)")

train_df.to_csv('binary_train_set.csv', index=False)
val_df.to_csv('binary_val_set.csv', index=False)
test_df.to_csv('binary_test_set.csv', index=False)
print("✓ Saved: binary_train_set.csv, binary_val_set.csv, binary_test_set.csv")


# ====================== STEP 5: DATASET CLASS ======================
class BinaryQualityDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['filename']
        label = int(row['binary_label'])
        img_path = os.path.join(self.img_dir, filename)

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            image = Image.new('RGB', (224, 224), color=(0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label


# ====================== STEP 6: TRANSFORMS ======================
print("\n[STEP 6] Preparing transforms...")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# ====================== STEP 7: DATALOADERS ======================
print("\n[STEP 7] Creating dataloaders...")

train_dataset = BinaryQualityDataset(train_df, IMG_DIR, train_transform)
val_dataset = BinaryQualityDataset(val_df, IMG_DIR, test_transform)
test_dataset = BinaryQualityDataset(test_df, IMG_DIR, test_transform)
full_dataset = BinaryQualityDataset(all_data_df, IMG_DIR, test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


# ====================== STEP 8: BINARY CNN MODEL ======================
print("\n[STEP 8] Initializing binary CNN model...")

class BinaryCNN(nn.Module):
    def __init__(self):
        super(BinaryCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Global pooling
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)  # Binary: Low/High
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = BinaryCNN().to(device)


# ====================== STEP 9: LOSS & OPTIMIZER ======================
print("\n[STEP 9] Setting up loss and optimizer...")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                       patience=5, factor=0.5)


# ====================== STEP 10: TRAINING ======================
print(f"\n[STEP 10] Training binary classifier for {EPOCHS} epochs...")

def train_binary_model(model, train_loader, val_loader, epochs, device):
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = outputs.max(1)
            train_total += labels.size(0)
            train_correct += preds.eq(labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100.0 * train_correct / train_total

        # Validate
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = outputs.max(1)
                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_binary_model.pth')
            print(f"Epoch {epoch+1}/{epochs} - NEW BEST! Val Acc: {val_acc:.2f}%")
        else:
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Train: {train_acc:.2f}%, Val: {val_acc:.2f}%")

    print(f"\nTraining complete. Best Val Acc: {best_val_acc:.2f}%")
    return history, best_val_acc

history, best_val_acc = train_binary_model(model, train_loader, val_loader, EPOCHS, device)


# ====================== STEP 11: TEST EVALUATION ======================
print("\n[STEP 11] Evaluating on test set...")

def evaluate_binary_model(model, loader, device, checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)

            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = 100.0 * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    print(f"Correct: {correct}/{total}")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds,
                                target_names=['Low Quality', 'High Quality']))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(all_labels, all_labels)
    print(cm)

    return acc, np.array(all_preds), np.array(all_labels)

test_acc, test_preds, test_labels = evaluate_binary_model(
    model, test_loader, device, 'best_binary_model.pth'
)


# ====================== STEP 12: APPLY TO ALL 531 ======================
print("\n[STEP 12] Applying binary classifier to all 531 images...")

def predict_full_binary(model, full_loader, device, checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, _ in full_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_probs)

full_preds, full_probs = predict_full_binary(
    model, full_loader, device, 'best_binary_model.pth'
)

all_data_df['predicted_binary'] = full_preds
all_data_df['binary_confidence'] = full_probs.max(axis=1)
all_data_df['quality_category'] = all_data_df['predicted_binary'].map({0: 'Low', 1: 'High'})

print("\nPredicted binary distribution (all images):")
pred_dist = all_data_df['quality_category'].value_counts()
for cat, cnt in pred_dist.items():
    print(f"  {cat:4s} Quality: {cnt:4d} ({100*cnt/len(all_data_df):5.2f}%)")


# ====================== STEP 13: FILTER HIGH QUALITY ======================
print("\n[STEP 13] Filtering high-quality images...")

CONF_THRESHOLD = 0.7

filtered_df = all_data_df[
    (all_data_df['predicted_binary'] == 1) &
    (all_data_df['binary_confidence'] >= CONF_THRESHOLD)
]

print(f"Original images: {len(all_data_df)}")
print(f"Filtered images (High quality, conf>=0.7): {len(filtered_df)}")
print(f"Retention rate: {100*len(filtered_df)/len(all_data_df):.2f}%")

all_data_df.to_csv('all_531_binary_predictions.csv', index=False)
filtered_df.to_csv('filtered_high_quality_binary.csv', index=False)

# Save history and report
pd.DataFrame(history).to_csv('binary_training_history.csv', index=False)

report = {
    'task': 'Binary Quality Classification',
    'blur_threshold': BLUR_THRESHOLD,
    'total_images': len(all_data_df),
    'split': {
        'train': len(train_df),
        'val': len(val_df),
        'test': len(test_df)
    },
    'best_val_acc': f"{best_val_acc:.2f}%",
    'test_acc': f"{test_acc:.2f}%",
    'predicted_distribution': pred_dist.to_dict(),
    'confidence_threshold': CONF_THRESHOLD,
    'filtered_images': len(filtered_df),
    'retention_rate': f"{100*len(filtered_df)/len(all_data_df):.2f}%"
}

with open('binary_quality_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("\n" + "="*70)
print("FILES SAVED")
print("="*70)
print("  best_binary_model.pth")
print("  binary_train_set.csv, binary_val_set.csv, binary_test_set.csv")
print("  all_531_binary_predictions.csv")
print("  filtered_high_quality_binary.csv")
print("  binary_training_history.csv")
print("  binary_quality_report.json")

print("\n" + "="*70)
print("BINARY CLASSIFICATION WORKFLOW COMPLETE!")
print("="*70)


BINARY QUALITY CLASSIFICATION - COMPLETE WORKFLOW

[STEP 1] Loading CSV...
Total images: 531

[STEP 2] Calculating blur scores...
Computing blur scores for all images...
Valid images: 531

Blur score statistics:
count      531.000000
mean      3268.799851
std       2438.882010
min          9.724937
25%        795.160729
50%       4110.863217
75%       5061.861892
max      13865.076692
Name: blur_score, dtype: float64

[STEP 3] Creating binary labels (threshold=3000)...

Binary distribution (full dataset):
  Low Quality (<3000):   232 (43.69%)
  High Quality (>=3000):  299 (56.31%)

[STEP 4] Splitting into train/val/test...
Train: 339 (63.8%)
Val:   85 (16.0%)
Test:  107 (20.2%)
✓ Saved: binary_train_set.csv, binary_val_set.csv, binary_test_set.csv

[STEP 6] Preparing transforms...

[STEP 7] Creating dataloaders...
Train: 339, Val: 85, Test: 107

[STEP 8] Initializing binary CNN model...
Using device: cuda

[STEP 9] Setting up loss and optimizer...

[STEP 10] Training binary classifier 

C:\Users\IISER\AppData\Local\Temp\ipykernel_12968\2455723290.py:305: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path))


Test Accuracy: 96.26%
Correct: 103/107

Classification Report:
              precision    recall  f1-score   support

 Low Quality       0.96      0.96      0.96        47
High Quality       0.97      0.97      0.97        60

    accuracy                           0.96       107
   macro avg       0.96      0.96      0.96       107
weighted avg       0.96      0.96      0.96       107


Confusion Matrix:
[[47  0]
 [ 0 60]]

[STEP 12] Applying binary classifier to all 531 images...


C:\Users\IISER\AppData\Local\Temp\ipykernel_12968\2455723290.py:349: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path))



Predicted binary distribution (all images):
  High Quality:  302 (56.87%)
  Low  Quality:  229 (43.13%)

[STEP 13] Filtering high-quality images...
Original images: 531
Filtered images (High quality, conf>=0.7): 288
Retention rate: 54.24%

FILES SAVED
  best_binary_model.pth
  binary_train_set.csv, binary_val_set.csv, binary_test_set.csv
  all_531_binary_predictions.csv
  filtered_high_quality_binary.csv
  binary_training_history.csv
  binary_quality_report.json

BINARY CLASSIFICATION WORKFLOW COMPLETE!
